# Data Science Coding Challenges — Runnable Examples

Run this notebook top to bottom. It includes Python, NumPy, Pandas, metrics, and ML-pipeline patterns using small synthetic examples.

Use `practice.md` for independent problems and `solution.md` for reference implementations.

In [ ]:
from collections import Counter
import math
import numpy as np
import pandas as pd


## 1. Python data structures

In [ ]:
def unique_in_order(items):
    seen, result = set(), []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

def two_sum(values, target):
    seen = {}
    for i, value in enumerate(values):
        if target - value in seen:
            return seen[target - value], i
        seen[value] = i
    return None

assert unique_in_order([3, 1, 3, 2, 1]) == [3, 1, 2]
assert two_sum([2, 7, 11, 15], 9) == (0, 1)
assert two_sum([], 4) is None
print("Python examples passed.")


## 2. NumPy vectorization

In [ ]:
arr = np.array([-3.0, 0.0, 2.0, 5.0])
clipped = np.maximum(arr, 0)
print("clipped:", clipped)

matrix = np.arange(12).reshape(3, 4)
print("shape:", matrix.shape)
print("row means:", matrix.mean(axis=1))
print("column means:", matrix.mean(axis=0))

left = np.ones((3, 1))
right = np.arange(4).reshape(1, 4)
assert (left + right).shape == (3, 4)


## 3. Pandas grouping and quality checks

In [ ]:
df = pd.DataFrame({
    "team": ["A", "A", "B", "B", "B"],
    "score": [10.0, np.nan, 8.0, 12.0, 9.0],
    "region": ["North", "North", "South", None, "South"]
})
df["team_mean"] = df.groupby("team")["score"].transform("mean")
quality = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
    "distinct": df.nunique(dropna=True)
})
display(df)
display(quality)


## 4. Merge validation

In [ ]:
left = pd.DataFrame({"department_id": [1, 2, 2], "employee": ["A", "B", "C"]})
right = pd.DataFrame({"department_id": [1, 2], "department": ["Data", "AI"]})
merged = left.merge(right, on="department_id", how="left", validate="many_to_one", indicator=True)
display(merged)
assert len(merged) == len(left)


## 5. Classification metrics from first principles

In [ ]:
def binary_metrics(y_true, y_pred):
    if len(y_true) != len(y_pred):
        raise ValueError("Inputs must have equal lengths")
    tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
    tn = sum(t == 0 and p == 0 for t, p in zip(y_true, y_pred))
    fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
    fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    accuracy = (tp + tn) / len(y_true) if y_true else 0.0
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

print(binary_metrics([0, 1, 1, 0, 1], [0, 1, 0, 0, 1]))
assert binary_metrics([1, 1], [1, 0])["recall"] == 0.5


## 6. Regression metrics

In [ ]:
def regression_metrics(y_true, y_pred):
    if len(y_true) != len(y_pred) or len(y_true) == 0:
        raise ValueError("Inputs must have equal, non-zero lengths")
    errors = [float(a) - float(p) for a, p in zip(y_true, y_pred)]
    return {
        "mae": sum(abs(e) for e in errors) / len(errors),
        "rmse": math.sqrt(sum(e * e for e in errors) / len(errors))
    }

print(regression_metrics([3, -0.5, 2, 7], [2.5, 0.0, 2, 8]))


## 7. A leakage-safe preprocessing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

numeric_features = ["age", "income"]
categorical_features = ["region", "plan"]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])
print("Pipeline defined. Fit it only after splitting data into train and test sets.")


## 8. Practice next

1. Implement `first_unique_char`.
2. Add edge-case tests to `minmax_scale`.
3. Create a top-three-per-group function with deterministic tie-breaking.
4. Modify the merge so a duplicate right-side key triggers validation.
5. Explain when stratified, group-based, or time-based validation is appropriate.

See `practice.md` for the full challenge list and `solution.md` for reference answers.